# Text-to-SQL PT-BR: QLoRA fine-tuning on Colab

Generates the dataset, trains all four configs, scores them and the untouched base model on the held-out test split, then writes `results/ablation_table.md` and `assets/loss_curves.png`.

Set `Runtime > Change runtime type > T4 GPU`, then `Run all`. The last cell downloads `results.zip`.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Clone and install

In [ ]:
import os

REPO = '/content/text2sql-ptbr-lora'
if not os.path.isdir(REPO):
    !git clone https://github.com/baconshow/text2sql-ptbr-lora.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt


def run(command):
    """Run a shell command and stop the notebook if it fails; a bare ! would carry on."""
    shell = get_ipython()
    shell.system(command)
    exit_code = shell.user_ns['_exit_code']
    if exit_code != 0:
        raise RuntimeError(f'Command failed with exit code {exit_code}: {command}')


## 2. Generate the dataset

In [ ]:
run('python src/data_prep.py --output data --n_samples 3000 --seed 42')

## 3. Train the four configs

In [ ]:
CONFIGS = {
    'base': 'base',
    'rank8': 'ablation_rank8',
    'rank32_allmodules': 'ablation_rank32_allmodules',
    'lr_high': 'ablation_lr_high',
}
for run_name, config in CONFIGS.items():
    if os.path.exists(f'runs/{run_name}/metrics.json'):
        print(f'{run_name}: already trained, skipping')
        continue
    run(f'python src/train.py --config configs/{config}.yaml')


## 4. Evaluate on the test split

No `--adapter` scores the base model as the baseline.

In [ ]:
BASE = 'Qwen/Qwen2.5-0.5B-Instruct'
run(f'python src/evaluate.py --data data/test.jsonl --base_model {BASE}')
for run_name in CONFIGS:
    run(f'python src/evaluate.py --adapter runs/{run_name} --data data/test.jsonl --base_model {BASE}')


## 5. Build the report

In [ ]:
run('python src/make_report.py --runs ' + ' '.join(CONFIGS))
from IPython.display import Image, Markdown, display
display(Markdown(open('results/ablation_table.md', encoding='utf-8').read()))
display(Image('assets/loss_curves.png'))


## 6. Download the results

Commit `results/` and `assets/loss_curves.png` to the repo.

In [ ]:
run('zip -q -r results.zip results assets/loss_curves.png runs/*/eval_results.json runs/*/metrics.json')
from google.colab import files
files.download('results.zip')


## 7. Optional: publish the adapter

Off by default so `Run all` never stops for a token. Set `PUSH_TO_HUB = True` to publish.

In [ ]:
PUSH_TO_HUB = False
if PUSH_TO_HUB:
    import getpass
    from huggingface_hub import create_repo, login, upload_folder
    login(token=getpass.getpass('HF token: '))
    repo_id = 'baconshow/qwen2.5-0.5b-text2sql-ptbr-lora'
    create_repo(repo_id, exist_ok=True)
    upload_folder(folder_path='runs/base', repo_id=repo_id, commit_message='Add LoRA adapter')
    print(f'Published: https://huggingface.co/{repo_id}')